In [1]:
%matplotlib inline
import torch
from torch import nn
from d2l import torch as d2l

先引入人工数据集
$$y = 0.05 + \sum_{i=1}^{d} 0.01 x_i + \varepsilon \quad \text{where } \varepsilon \sim \mathcal{N}(0, 0.01^2)$$

In [2]:
n_train, n_test, num_inputs, batch_size = 20, 100, 200, 5
true_w, true_b = torch.ones((num_inputs, 1)) * 0.01, 0.05
train_data = d2l.synthetic_data(true_w, true_b, n_train)
train_iter = d2l.load_array(train_data, batch_size)
test_data = d2l.synthetic_data(true_w, true_b, n_test)
test_iter = d2l.load_array(test_data, batch_size, is_train=False)

初始化模型参数

In [3]:
def init_params():
    w=torch.normal(0,1,size=(num_inputs,1),requires_grad=True)
    b=torch.zeros(1,requires_grad=True)
    return w,b

定义L2范数惩罚

In [4]:
def L2_penalty(w):
    return torch.sum(torch.pow(w,2))/2

定义训练函数实现

In [5]:
def train(lambd):
    w,b = init_params()
    net,loss=lambda x:d2l.linreg(x,w,b),d2l.squared_loss
    num_epochs,lr=100,0.03
    for epoch in range(num_epochs):
        for x,y in train_iter:
            l=loss(net(x),y)+lambd*L2_penalty(w)
            l.sum().backward()
            d2l.sgd([w,b],lr,batch_size)
        if (epoch +1)%20==0:
            train_loss=d2l.evaluate_loss(net,train_iter,loss)
            test_loss=d2l.evaluate_loss(net,test_iter,loss)
            print(f'epoch {epoch+1:3d} | train loss {train_loss:.4f} | test loss {test_loss:.4f} | w L2范数 {torch.norm(w.detach()).item():.4f}')
    print()
    print('训练完成，w的L2范数是：',format(torch.norm(w).item(),'.4f'))


In [6]:
train(lambd=10)

epoch  20 | train loss 0.0035 | test loss 0.0096 | w L2范数 0.0325
epoch  40 | train loss 0.0022 | test loss 0.0074 | w L2范数 0.0315
epoch  60 | train loss 0.0033 | test loss 0.0072 | w L2范数 0.0353
epoch  80 | train loss 0.0023 | test loss 0.0074 | w L2范数 0.0346
epoch 100 | train loss 0.0033 | test loss 0.0077 | w L2范数 0.0265

训练完成，w的L2范数是： 0.0265


C:\Users\linhaoming\miniconda3\envs\d2l\Lib\site-packages\d2l\torch.py:3179: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  self.data = [a + float(b) for a, b in zip(self.data, args)]


简介实现

In [7]:
def train_concise(wd):
    net = nn.Sequential(nn.Linear(num_inputs, 1))
    for param in net.parameters():
        param.data.normal_()
    loss=nn.MSELoss()
    num_epochs, lr = 100, 0.003
    trainer=torch.optim.SGD([{"params":net[0].weight,"weight_decay":wd},{"params":net[0].bias}], lr=lr)
    for epoch in range(num_epochs):
        for x, y in train_iter:
            trainer.zero_grad()
            l = loss(net(x), y)
            l.backward()
            trainer.step()
        if (epoch+1)%20==0:
            train_loss=d2l.evaluate_loss(net,train_iter,loss)
            test_loss=d2l.evaluate_loss(net,test_iter,loss)
            print(f'epoch {epoch+1:3d} | train loss {train_loss:.4f} | test loss {test_loss:.4f}')
print('简洁实现，wd=10：')
train_concise(10)


简洁实现，wd=10：
epoch  20 | train loss 0.0083 | test loss 1.6091
epoch  40 | train loss 0.0065 | test loss 0.0685
epoch  60 | train loss 0.0053 | test loss 0.0493
epoch  80 | train loss 0.0044 | test loss 0.0399
epoch 100 | train loss 0.0037 | test loss 0.0329
